# 固定观测边界诊断：e01/e02 × P0/PM

对已完成三组结果做事后真值诊断。选择 Run all；第一个可执行单元只挂载 Drive，随后自动加载 GitHub 的 dev/local-joint-state-payload-v1 分支源码，无需上传 ZIP 或填写参数。也可选用第二代码单元的 SOURCE_DIRECTORY 或 SOURCE_ZIP。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. 加载已保存输入与 B 分支源码

输入：B-Terminal-Three-Group-V1/fixed-four-new-v1。新输出：B-State-Observation-Boundaries-V1/fixed-e01-e02-v1（均在 MyDrive/Video-WM 下）。已有执行记录时跳过源码下载和安装，直接只读恢复。


In [ ]:

from pathlib import Path
import json, os, stat, subprocess, sys, tempfile, time, urllib.request, zipfile
INPUT = Path('/content/drive/MyDrive/Video-WM/B-Terminal-Three-Group-V1/fixed-four-new-v1')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/B-State-Observation-Boundaries-V1/fixed-e01-e02-v1')
SOURCE_DIRECTORY = None  # Optional existing directory containing the new runner.
SOURCE_ZIP = None  # Optional ordinary local ZIP; the default downloads the B branch.
SOURCE_REF = 'dev/local-joint-state-payload-v1'
SOURCE_URL = 'https://codeload.github.com/RICHAAARC/SC-SSTW/zip/refs/heads/' + SOURCE_REF
CASES = ({'source': 'b2-m3-e01', 'a': 17, 's': 19, 'phase': 2, 'start': 72},
         {'source': 'b2-m3-e02', 'a': 6, 's': 16, 'phase': 1, 'start': 60})

def make_plan(input_directory=None):
    from pathlib import Path
    source_root = Path(input_directory or '/content/drive/MyDrive/Video-WM/B-Terminal-Three-Group-V1/fixed-four-new-v1')
    units = {}
    for case in CASES:
        source,start,s,phase=(case[k] for k in ('source','start','s','phase'))
        for arm in ('P0', 'PM'):
            for layer in ('N', 'A', 'B', 'C', 'D'):
                slot = f'{source}/{arm}/{layer}'
                if layer == 'N': relative = f'private/{source}/native/{arm}_Q.json'
                elif layer == 'D': relative = f'observations/{source}/{arm}/CROP/M/p{phase}/key00/Q.json'
                else: relative = f'media/{source}/{arm}/full/'+('rgb8.pt' if layer == 'C' else 'float_rgb.pt')
                units[slot] = dict(slot=slot, source=source, arm=arm, layer=layer,
                    s=s, phase=phase, source_start=start, input_path=str(source_root/relative),
                    source_frames=[0,180] if layer == 'A' else [start,start+40],
                    encoder_frames=181 if layer == 'A' else 41 if layer in ('B','C') else 0,
                    kind='E181' if layer == 'A' else 'E41' if layer in ('B','C') else 'SAVED_Q',
                    status='NOT_ENTERED', attempts=0, q40=[None]*40, metrics=None)
    return dict(input_directory=str(source_root), units=units,
        denominators=dict(cases=4, layers=5, layer_cases=20, coordinates=800,
                          new_encodes=12, E181=4, E41=8, new_input_frames=1052,
                          new_Q40=12, reused_Q40=8, new_coordinates=480),
        post_truth_only=True, blind_receiver_changed=False)

def execution_artifacts(output):
    from pathlib import Path
    output = Path(output)
    names = ('attempt.json','plan.json','ledger.json','result.json','evaluation.json',
             'inputs.json','physical_counts.json','release.json')
    found = [n+s for n in names for s in ('','.tmp') if (output/(n+s)).exists() or (output/(n+s)).is_symlink()]
    for name in ('receipts','layers'):
        folder = output/name
        if folder.is_dir():
            found.extend(str(p.relative_to(output)) for p in folder.rglob('*') if p.is_file() or p.is_symlink())
    return sorted(found)

def recover_output(output):
    """Read-only receipt overlay, embedded in the notebook; never resumes work."""
    from pathlib import Path
    from collections import Counter
    import json
    output = Path(output)
    errors = {}
    def load(name, default):
        try: return json.loads((output/name).read_text(encoding='utf-8'))
        except (OSError, ValueError) as error:
            errors[name] = str(error)
            return default
    plan = load('plan.json', make_plan())
    if not isinstance(plan, dict) or not isinstance(plan.get('units'), dict):
        errors['plan.json'] = 'invalid saved plan'
        plan = make_plan()
    layers = {slot: dict(row) for slot,row in make_plan(plan.get('input_directory'))['units'].items()}
    ledger = load('ledger.json', {})
    if not isinstance(ledger, dict):
        errors['ledger.json'] = 'invalid ledger'
        ledger = {}
    for path in sorted((output/'receipts').rglob('*.json'))+sorted((output/'receipts').rglob('*.json.tmp')):
        rel = path.relative_to(output/'receipts').as_posix()
        slot = rel[:-9] if rel.endswith('.tmp') else rel[:-5]
        old = ledger.get(slot, {})
        try:
            receipt = json.loads(path.read_text(encoding='utf-8'))
            if rel.endswith('.tmp'): raise ValueError('uncommitted receipt')
            if not isinstance(receipt, dict) or receipt.get('slot') != slot: raise ValueError('receipt slot mismatch')
            status, attempt = receipt.get('status'), receipt.get('attempts')
            if status not in ('NOT_ENTERED','STARTED','RETURNED','FAILED','INTERRUPTED'): raise ValueError('invalid receipt status')
            if type(attempt) is not int or attempt != (0 if status == 'NOT_ENTERED' else 1): raise ValueError('invalid receipt attempts')
            if isinstance(old, dict) and old.get('attempts') == 1 and attempt == 0: raise ValueError('cannot erase spent attempt')
            ledger[slot] = receipt
        except (OSError, ValueError) as error:
            errors['receipts/'+rel] = str(error)
            ledger[slot] = dict(slot=slot,status='UNRESOLVED_RECEIPT',attempts=old.get('attempts') if isinstance(old,dict) else None)
    for slot,row in layers.items():
        receipt = ledger.get(slot, {})
        if not isinstance(receipt,dict): receipt = dict(status='UNRESOLVED_RECEIPT',attempts=None)
        if receipt.get('status') == 'RETURNED':
            value = load('layers/'+slot+'.json', None)
            if isinstance(value,dict) and isinstance(value.get('q40'),list) and len(value['q40']) == 40:
                row.update(value)
            else:
                row.update(status='UNRESOLVED_RESULT', attempts=1)
        else:
            partial = output/'layers'/(slot+'.json')
            if partial.is_file():
                value = load('layers/'+slot+'.json', None)
                if isinstance(value,dict) and isinstance(value.get('q40'),list) and len(value['q40'])==40:
                    row.update(value)
            row.update({k:v for k,v in receipt.items() if k not in ('q40','metrics')})
    return dict(status='READ_ONLY_EXISTING_ATTEMPT', physical_reentry=False,
        denominators=make_plan()['denominators'], layers=layers, ledger=ledger,
        counts=dict(Counter(r['status'] for r in layers.values())),
        physical_counts=load('physical_counts.json', None),
        evaluation=load('evaluation.json', None), read_errors=errors)
EXISTING = bool(execution_artifacts(OUTPUT))
SOURCE = None
if EXISTING:
    print('Existing fixed attempt: read-only. No source download, setup, model load or encoding.')
else:
    if SOURCE_DIRECTORY:
        SOURCE = Path(SOURCE_DIRECTORY).expanduser().resolve()
    else:
        workspace = Path(tempfile.mkdtemp(prefix='b-state-observation-source-')).resolve()
        if SOURCE_ZIP is None:
            archive_path = workspace/'source.zip'
            print('Download available B source:', SOURCE_URL, flush=True)
            urllib.request.urlretrieve(SOURCE_URL, archive_path)
        else:
            archive_path = Path(SOURCE_ZIP).expanduser().resolve()
        target = workspace/'source'
        target.mkdir()
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                normalized = member.filename.replace('\\','/')
                parts = normalized.split('/')
                resolved = (target/normalized).resolve()
                if normalized.startswith('/') or any(':' in part for part in parts) or not resolved.is_relative_to(target):
                    raise ValueError('Unsafe ZIP path: '+member.filename)
                if stat.S_ISLNK(member.external_attr >> 16):
                    raise ValueError('ZIP symbolic links are not supported: '+member.filename)
            archive.extractall(target)
        candidates = [p.parents[2] for p in target.rglob('b_state_observation_boundaries_v1_run.py')
                      if p.parent.name == 'wan_state_clock' and p.parent.parent.name == 'experiments']
        if len(candidates) != 1:
                raise ValueError('The source archive must contain one diagnostic entry point.')
        SOURCE = candidates[0]
    entry = SOURCE/'experiments/wan_state_clock/b_state_observation_boundaries_v1_run.py'
    if not entry.is_file():
        raise FileNotFoundError('New diagnostic runner missing: '+str(entry))
    OUTPUT.mkdir(parents=True,exist_ok=True)
    print('Source directory:',SOURCE)
    print('Original saved input:',INPUT)
    print('Fixed new output:',OUTPUT)
    print('Maximum 12 VAE encodes; first failure stops. Repeating Run all is read-only.')


## 2. 执行固定的一次预算

最多 12 次 VAE 编码：4 次 E181 + 8 次 E41，共 1052 输入帧。N/D 复用旧 Q；A/B/C 新编码。首个失败或中断即停止，后续行保留 NOT_ENTERED；重复 Run all 不补跑。首次使用沿用已跑通的依赖修复与 FP32 VAE。


In [ ]:
if EXISTING:
    print('Reading existing attempt; no physical work.')
else:
    LOG = OUTPUT/'setup.log'
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, cwd=SOURCE, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode
    def ensure_imports(probe, packages, label):
        import re
        mapping = {p.replace('-', '_'):p for p in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1).split('.')[0] in mapping:
                package = mapping[missing.group(1).split('.')[0]]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' actual import failure; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, action)
    ensure_imports('import numpy,torch', ['numpy','torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub,transformers,sentencepiece,ftfy; from diffusers import AutoencoderKLWan',
        ['diffusers','accelerate','safetensors','huggingface-hub','transformers','sentencepiece','ftfy'], 'Wan VAE')

    command([sys.executable,'-c',
        'import runtime.wan.b_state_observation_boundaries_v1 as r; '
        'import runtime.wan.b_state_observation_boundaries_backend_v1 as b; '
        'print(r.__file__); print(b.__file__)'], 'Probe actual source imports')
    args = [sys.executable,'-m','experiments.wan_state_clock.b_state_observation_boundaries_v1_run',
            '--input',str(INPUT),'--output',str(OUTPUT)]
    log = OUTPUT/'run.log'
    with log.open('a') as stream:
        worker = subprocess.Popen(args,cwd=SOURCE,stdout=stream,stderr=subprocess.STDOUT)
        try:
            while worker.poll() is None:
                try:
                    worker.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    progress = OUTPUT/'progress.json'
                    print(progress.read_text() if progress.is_file() else 'Worker starting; log: '+str(log),flush=True)
        except BaseException:
            worker.terminate()
            try:worker.wait(timeout=10)
            except subprocess.TimeoutExpired:
                worker.kill();worker.wait()
            raise
    print('Worker exit:',worker.returncode,'Log:',log)
    if worker.returncode:
        print('Attempt stopped. The next cell reads retained rows; Run all does not retry.')


## 3. 查看保留的全部结果

固定 20 个层/案例、800 个坐标。显示 Q40 分解和 PM-P0 配对方向/传递比；失败和零分母明确保留。tau 仅为参考线。本次使用已知真值定位固定窗口，不是盲恢复成功，也不能单独证明普遍因果。


In [ ]:

report = recover_output(OUTPUT)
print(json.dumps({k:report[k] for k in ('status','denominators','counts','physical_counts','read_errors')},indent=2))
for slot,row in report['layers'].items():
    metric=row.get('metrics') or {}
    print(slot,row['status'],'attempts=',row.get('attempts'),
          'S=',metric.get('S'),'signed alpha=',metric.get('alpha_signed'),
          'residual energy=',metric.get('residual_energy'))
evaluation=report.get('evaluation') or {}
for slot,row in evaluation.get('pairs',{}).items():
    print('PM-P0',slot,row['status'],'raw direction=',row.get('cosine_native'),'mean gain=',row.get('mean_gain'))
print('Reference line only:',evaluation.get('tau_reference_only'))
print('Descriptive first loss:',evaluation.get('first_reference_line_loss_boundary'))
print('Saved outputs:',OUTPUT)
print('Fixed 20 layer/case rows, 800 coordinates. This is post-truth diagnosis, not blind detection or scientific PASS.')
